# 📊 MÓDULO 4: Evaluación de Modelos
## Métricas, Matriz de Confusión y Comparación Final

**Objetivo:** Evaluar ambos modelos y decidir cuál recomendar.

---

## 4.1 Configuración y Entrenamiento Rápido

In [ ]:
import sys
import time

EN_COLAB = 'google.colab' in sys.modules
RUTA_DATOS = '/content/drive/MyDrive/ML_Vuelos/data/' if EN_COLAB else '../data/'

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Vuelos_Modulo4").config("spark.driver.memory", "4g").getOrCreate()

# Preparar datos
from pyspark.ml.feature import StringIndexer, VectorAssembler
df = spark.read.csv(RUTA_DATOS + "flights_2015_full.csv", header=True, inferSchema=True)

for col_name in ["AIRLINE", "ORIGIN", "DEST"]:
    indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    df = indexer.fit(df).transform(df)

feature_cols = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "DISTANCE", "AIRLINE_idx", "ORIGIN_idx", "DEST_idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assembler.transform(df).select("features", "DEP_DEL15")
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

# Entrenar modelos
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier

print("🌳 Entrenando Árbol...")
dt = DecisionTreeClassifier(labelCol="DEP_DEL15", featuresCol="features", maxDepth=5)
modelo_arbol = dt.fit(train_df)

print("🌲 Entrenando Bosque...")
rf = RandomForestClassifier(labelCol="DEP_DEL15", featuresCol="features", numTrees=100, maxDepth=5, seed=42)
modelo_bosque = rf.fit(train_df)

print("✅ Modelos listos")

## 4.2 Predicciones

In [ ]:
# Hacer predicciones en test
pred_arbol = modelo_arbol.transform(test_df)
pred_bosque = modelo_bosque.transform(test_df)

print("📊 Predicciones del Árbol:")
pred_arbol.select("DEP_DEL15", "prediction", "probability").show(5)

## 4.3 Métricas de Evaluación

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Evaluadores
eval_acc = MulticlassClassificationEvaluator(labelCol="DEP_DEL15", metricName="accuracy")
eval_f1 = MulticlassClassificationEvaluator(labelCol="DEP_DEL15", metricName="f1")
eval_auc = BinaryClassificationEvaluator(labelCol="DEP_DEL15", metricName="areaUnderROC")

# Calcular métricas
metricas = {
    "Árbol de Decisión": {
        "Accuracy": eval_acc.evaluate(pred_arbol),
        "F1-Score": eval_f1.evaluate(pred_arbol),
        "AUC-ROC": eval_auc.evaluate(pred_arbol)
    },
    "Random Forest": {
        "Accuracy": eval_acc.evaluate(pred_bosque),
        "F1-Score": eval_f1.evaluate(pred_bosque),
        "AUC-ROC": eval_auc.evaluate(pred_bosque)
    }
}

print("\n" + "="*60)
print("📊 COMPARACIÓN DE MÉTRICAS")
print("="*60)
print(f"{'Métrica':<15} {'Árbol':>15} {'Random Forest':>15} {'Ganador':>12}")
print("-"*60)

for metrica in ["Accuracy", "F1-Score", "AUC-ROC"]:
    val_arbol = metricas["Árbol de Decisión"][metrica]
    val_bosque = metricas["Random Forest"][metrica]
    ganador = "🌲 Bosque" if val_bosque > val_arbol else "🌳 Árbol"
    print(f"{metrica:<15} {val_arbol:>14.4f} {val_bosque:>15.4f} {ganador:>12}")

print("="*60)

## 4.4 Matriz de Confusión

In [ ]:
from pyspark.sql.functions import col, count, when

def matriz_confusion(predicciones, nombre):
    print(f"\n📊 MATRIZ DE CONFUSIÓN - {nombre}")
    print("="*50)
    
    # Calcular componentes
    tp = predicciones.filter((col("DEP_DEL15") == 1) & (col("prediction") == 1)).count()
    tn = predicciones.filter((col("DEP_DEL15") == 0) & (col("prediction") == 0)).count()
    fp = predicciones.filter((col("DEP_DEL15") == 0) & (col("prediction") == 1)).count()
    fn = predicciones.filter((col("DEP_DEL15") == 1) & (col("prediction") == 0)).count()
    
    print(f"\n                    PREDICCIÓN")
    print(f"                 Puntual  Retrasado")
    print(f"         Puntual   {tn:>7,}   {fp:>7,}")
    print(f"REAL")
    print(f"       Retrasado   {fn:>7,}   {tp:>7,}")
    
    print(f"\n📌 Interpretación:")
    print(f"   ✅ True Positives (TP):  {tp:>10,} - Retrasos correctamente predichos")
    print(f"   ✅ True Negatives (TN):  {tn:>10,} - Puntuales correctamente predichos")
    print(f"   ❌ False Positives (FP): {fp:>10,} - Falsas alarmas")
    print(f"   ❌ False Negatives (FN): {fn:>10,} - Retrasos NO detectados")
    
    # Métricas derivadas
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f"\n📈 Métricas derivadas:")
    print(f"   Precision: {precision:.4f} - De los que predije retraso, ¿cuántos acerté?")
    print(f"   Recall:    {recall:.4f} - De los retrasos reales, ¿cuántos detecté?")
    
    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}

cm_arbol = matriz_confusion(pred_arbol, "ÁRBOL DE DECISIÓN")
cm_bosque = matriz_confusion(pred_bosque, "RANDOM FOREST")

## 4.5 Análisis de Costos (para Finanzas)

In [ ]:
# Costo estimado por tipo de error
COSTO_FN = 150  # Costo de un retraso NO predicho (compensación)
AHORRO_TP = 100  # Ahorro por predecir correctamente un retraso

print("\n💰 ANÁLISIS FINANCIERO")
print("="*55)
print(f"Supuestos: Costo por FN = ${COSTO_FN} | Ahorro por TP = ${AHORRO_TP}")
print("-"*55)

for nombre, cm in [("Árbol", cm_arbol), ("Bosque", cm_bosque)]:
    ahorro = cm["TP"] * AHORRO_TP
    perdida = cm["FN"] * COSTO_FN
    balance = ahorro - perdida
    
    print(f"\n{nombre}:")
    print(f"  💚 Ahorro (TP × ${AHORRO_TP}):  ${ahorro:>12,}")
    print(f"  💔 Pérdida (FN × ${COSTO_FN}): ${perdida:>12,}")
    print(f"  {'📈' if balance > 0 else '📉'} Balance neto:       ${balance:>12,}")

## 4.6 Reglas del Árbol (para Marketing)

In [ ]:
# Extraer reglas interpretables del árbol
print("\n📢 REGLAS PARA SISTEMA DE ALERTAS")
print("="*50)
print("\nBasado en el árbol de decisión, estas son las condiciones")
print("que más aumentan la probabilidad de retraso:\n")

# Mostrar estructura simplificada
reglas = modelo_arbol.toDebugString
lineas = reglas.split("\n")[:30]
for linea in lineas:
    print(linea)

## 4.7 Tabla Comparativa Final

In [ ]:
print("\n" + "="*70)
print("🏆 TABLA COMPARATIVA FINAL")
print("="*70)
print(f"{'Criterio':<25} {'Árbol':>20} {'Random Forest':>20}")
print("-"*70)
print(f"{'Accuracy':<25} {metricas['Árbol de Decisión']['Accuracy']:>19.2%} {metricas['Random Forest']['Accuracy']:>19.2%}")
print(f"{'F1-Score':<25} {metricas['Árbol de Decisión']['F1-Score']:>19.4f} {metricas['Random Forest']['F1-Score']:>19.4f}")
print(f"{'AUC-ROC':<25} {metricas['Árbol de Decisión']['AUC-ROC']:>19.4f} {metricas['Random Forest']['AUC-ROC']:>19.4f}")
print(f"{'Interpretabilidad':<25} {'✅ Alta':>20} {'❌ Baja':>20}")
print(f"{'Velocidad':<25} {'✅ Rápido':>20} {'⚠️ Lento':>20}")
print(f"{'Producción (tiempo real)':<25} {'✅ Recomendado':>20} {'⚠️ Evaluar':>20}")
print("="*70)

---
## ✅ CHECKPOINT MÓDULO 4 - FINAL

Ahora puedes responder las preguntas de TU MISIÓN usando:

- Métricas de evaluación
- Matriz de confusión
- Feature importance
- Reglas del árbol
- Análisis de costos

**¡Prepara tu recomendación para el CEO!** 🎯